# Task 1 — Data Exploration and Enrichment
### Ethiopia Financial Inclusion Forecasting — Selam Analytics

**Objective:** Understand the starter dataset's unified schema, explore its structure, and enrich it
with additional real, sourced data useful for forecasting Access and Usage.

**Sections**
1. Load the three starter files
2. Understand the schema
3. Explore the data (counts, temporal range, indicators, events, impact_links)
4. Enrich the dataset (new observations, events, impact_links — all sourced)
5. Save the enriched dataset


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)


## 1. Load the three starter files

In [2]:
df = pd.read_csv('../data/raw/ethiopia_fi_unified_data.csv')
ref = pd.read_csv('../data/raw/reference_codes.csv')

print(f"Unified dataset: {df.shape[0]} records x {df.shape[1]} columns")
print(f"Reference codes: {ref.shape[0]} rows")
df.head()


Unified dataset: 57 records x 35 columns
Reference codes: 71 rows


,record_id,parent_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,EVT_0001,NaN,event,product_launch,NaN,Telebirr Launch,EVT_TELEBIRR,NaN,NaN,Launched,categorical,NaN,2021-05-17,NaN,NaN,2021,all,national,NaN,Ethio Telecom,operator,https://www.ethiotelecom.et/,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,First major mobile money service in Ethiopia,NaN
1,EVT_0002,NaN,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,NaN,NaN,Launched,categorical,NaN,2022-08-01,NaN,NaN,2022,all,national,NaN,News,news,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,End of state telecom monopoly,NaN
2,EVT_0003,NaN,event,product_launch,NaN,M-Pesa Ethiopia Launch,EVT_MPESA,NaN,NaN,Launched,categorical,NaN,2023-08-01,NaN,NaN,2023,all,national,NaN,Safaricom,operator,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Second mobile money entrant,NaN
3,EVT_0004,NaN,event,infrastructure,NaN,Fayda Digital ID Program Rollout,EVT_FAYDA,NaN,NaN,Launched,categorical,NaN,2024-01-01,NaN,NaN,2024,all,national,NaN,NIDP,regulator,https://www.id.gov.et/,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,National biometric digital ID system,NaN
4,EVT_0005,NaN,event,policy,NaN,Foreign Exchange Liberalization,EVT_FX_REFORM,NaN,NaN,Implemented,categorical,NaN,2024-07-29,NaN,NaN,2024,all,national,NaN,NBE,regulator,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Birr float introduced,NaN


## 2. Understand the schema

The dataset uses a **unified schema**: every record type (`observation`, `event`, `impact_link`, `target`)
shares the same 34 columns, and `record_type` tells us how to interpret each row.

Key design principle (from `SCHEMA_DESIGN.md`): **events are not pre-assigned to a pillar.** An event
like Telebirr's launch might affect both Access and Usage, so forcing a single pillar onto the event
itself would bias the analysis. Instead, an event's effects are captured through separate
`impact_link` records that connect back to the event via `parent_id`, each with its own pillar.


In [3]:
df['record_type'].value_counts()


record_type
observation    30
impact_link    14
event          10
target          3
Name: count, dtype: int64

In [4]:
# All valid field values, for reference
ref.head(20)


,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (links via pare...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All
5,record_type,forecast,Predicted future value,All
6,category,product_launch,New product or service introduced,event
7,category,market_entry,New competitor enters market,event
8,category,market_exit,Competitor leaves market,event
9,category,policy,Government strategy or regulatory framework,event


### How `impact_link` connects to `event` via `parent_id`

In [5]:
sample_event = df[df['record_type'] == 'event'].iloc[0]
print("EVENT:", sample_event['record_id'], '-', sample_event['indicator'])

linked = df[(df['record_type'] == 'impact_link') & (df['parent_id'] == sample_event['record_id'])]
print(f"\nImpact links pointing to this event: {len(linked)}")
linked[['record_id', 'parent_id', 'pillar', 'related_indicator', 'impact_direction', 'impact_magnitude', 'lag_months']]


EVENT: EVT_0001 - Telebirr Launch

Impact links pointing to this event: 3


,record_id,parent_id,pillar,related_indicator,impact_direction,impact_magnitude,lag_months
10,IMP_0001,EVT_0001,ACCESS,ACC_OWNERSHIP,increase,high,12.0
11,IMP_0002,EVT_0001,USAGE,USG_TELEBIRR_USERS,increase,high,3.0
12,IMP_0003,EVT_0001,USAGE,USG_P2P_COUNT,increase,high,6.0


## 3. Explore the data

### 3.1 Counts by record_type, pillar, source_type, confidence

In [6]:
print("By record_type:")
print(df['record_type'].value_counts(), "\n")

print("By pillar:")
print(df['pillar'].value_counts(dropna=False), "\n")

print("By source_type:")
print(df['source_type'].value_counts(dropna=False), "\n")

print("By confidence:")
print(df['confidence'].value_counts(dropna=False))


By record_type:
record_type
observation    30
impact_link    14
event          10
target          3
Name: count, dtype: int64 

By pillar:
pillar
ACCESS           20
USAGE            17
NaN              10
GENDER            6
AFFORDABILITY     4
Name: count, dtype: int64 

By source_type:
source_type
operator      15
NaN           14
survey        10
regulator      7
research       4
policy         3
news           2
calculated     2
Name: count, dtype: int64 

By confidence:
confidence
high      44
medium    13
Name: count, dtype: int64


### 3.2 Temporal range of observations

In [7]:
obs = df[df['record_type'] == 'observation'].copy()
obs['observation_date'] = pd.to_datetime(obs['observation_date'])

print(f"Earliest observation: {obs['observation_date'].min().date()}")
print(f"Latest observation:   {obs['observation_date'].max().date()}")
print(f"Span: {(obs['observation_date'].max() - obs['observation_date'].min()).days / 365.25:.1f} years")

obs.groupby(obs['observation_date'].dt.year).size().rename('n_observations')


Earliest observation: 2014-12-31
Latest observation:   2025-12-31
Span: 11.0 years


observation_date
2014     1
2017     1
2021     5
2023     1
2024    11
2025    11
Name: n_observations, dtype: int64

### 3.3 Unique indicators and their coverage

In [8]:
indicator_coverage = (
    obs.groupby('indicator_code')
    .agg(indicator=('indicator', 'first'),
         pillar=('pillar', 'first'),
         n_obs=('record_id', 'count'),
         first_date=('observation_date', 'min'),
         last_date=('observation_date', 'max'))
    .sort_values('pillar')
)
indicator_coverage


,indicator,pillar,n_obs,first_date,last_date
indicator_code,,,,,
ACC_4G_COV,4G Population Coverage,ACCESS,2,2023-06-30,2025-06-30
ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS,3,2024-08-15,2025-05-15
ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS,2,2021-12-31,2024-11-29
ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS,1,2025-12-31,2025-12-31
ACC_OWNERSHIP,Account Ownership Rate,ACCESS,6,2014-12-31,2024-11-29
AFF_DATA_INCOME,Data Affordability Index,AFFORDABILITY,1,2024-12-31,2024-12-31
GEN_GAP_ACC,Account Ownership Gender Gap,GENDER,2,2021-12-31,2024-11-29
GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER,1,2024-12-31,2024-12-31
GEN_MM_SHARE,Female Mobile Money Account Share,GENDER,1,2024-12-31,2024-12-31


### 3.4 Cataloged events and their dates

In [9]:
events = df[df['record_type'] == 'event'].copy()
events['observation_date'] = pd.to_datetime(events['observation_date'])
events[['record_id', 'category', 'indicator', 'observation_date', 'confidence']].sort_values('observation_date')


,record_id,category,indicator,observation_date,confidence
0,EVT_0001,product_launch,Telebirr Launch,2021-05-17,high
8,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01,high
1,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01,high
2,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01,high
3,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01,high
4,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29,high
5,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01,high
6,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27,high
9,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15,high
7,EVT_0008,infrastructure,EthioPay Instant Payment System Launch,2025-12-18,high


### 3.5 Existing impact_links: which events affect which indicators

In [10]:
impact_links = df[df['record_type'] == 'impact_link'].copy()

# join event name onto each impact_link for readability
event_names = events.set_index('record_id')['indicator']
impact_links['event_name'] = impact_links['parent_id'].map(event_names)

impact_links[['record_id', 'parent_id', 'event_name', 'pillar', 'related_indicator',
              'relationship_type', 'impact_direction', 'impact_magnitude', 'lag_months', 'evidence_basis']]


,record_id,parent_id,event_name,pillar,related_indicator,relationship_type,impact_direction,impact_magnitude,lag_months,evidence_basis
10,IMP_0001,EVT_0001,Telebirr Launch,ACCESS,ACC_OWNERSHIP,direct,increase,high,12.0,literature
11,IMP_0002,EVT_0001,Telebirr Launch,USAGE,USG_TELEBIRR_USERS,direct,increase,high,3.0,empirical
12,IMP_0003,EVT_0001,Telebirr Launch,USAGE,USG_P2P_COUNT,direct,increase,high,6.0,empirical
13,IMP_0004,EVT_0002,Safaricom Ethiopia Commercial Launch,ACCESS,ACC_4G_COV,direct,increase,medium,12.0,empirical
14,IMP_0005,EVT_0002,Safaricom Ethiopia Commercial Launch,AFFORDABILITY,AFF_DATA_INCOME,indirect,decrease,medium,12.0,literature
15,IMP_0006,EVT_0003,M-Pesa Ethiopia Launch,USAGE,USG_MPESA_USERS,direct,increase,high,3.0,empirical
16,IMP_0007,EVT_0003,M-Pesa Ethiopia Launch,ACCESS,ACC_MM_ACCOUNT,direct,increase,medium,6.0,theoretical
17,IMP_0008,EVT_0004,Fayda Digital ID Program Rollout,ACCESS,ACC_OWNERSHIP,enabling,increase,medium,24.0,literature
18,IMP_0009,EVT_0004,Fayda Digital ID Program Rollout,GENDER,GEN_GAP_ACC,indirect,decrease,medium,24.0,literature
19,IMP_0010,EVT_0005,Foreign Exchange Liberalization,AFFORDABILITY,AFF_DATA_INCOME,indirect,increase,high,3.0,empirical


In [11]:
# Which events currently have NO modeled impact_link at all?
events_with_links = set(impact_links['parent_id'])
all_events = set(events['record_id'])
uncovered = all_events - events_with_links

print("Events with zero impact_links in the base dataset:")
for e in uncovered:
    print(" -", e, '-', events.set_index('record_id').loc[e, 'indicator'])


Events with zero impact_links in the base dataset:
 - EVT_0006 - P2P Transaction Count Surpasses ATM
 - EVT_0009 - NFIS-II Strategy Launch


**Insight:** `EVT_0009` (NFIS-II strategy) has no modeled impact_link in the base 14, despite being the
policy source of the 70%-by-2025 account-ownership target (`REC_0031`). This is one of the gaps we address
in enrichment below.

## 4. Enrich the dataset

Enrichment logic lives in `../src/enrich_data.py` so it's reproducible and reviewable as code, not just
notebook output. It adds:

- **12 new observations** — smartphone penetration & its gender gap, gender-disaggregated mobile
  ownership, adult literacy, electricity access, 2021 gender-disaggregated digital payment baseline,
  EthSwitch Instant Payment System (IPS) transaction volume, and IPS cost reduction.
- **3 new events** — EthSwitch IPS go-live (Feb 2024), NBE's ETHQR standard approval (Apr 2024), and
  the ETHQR adoption mandate (Nov 2024) — none of which were captured in the original 10 events, despite
  being arguably as structurally important as Telebirr's launch.
- **5 new impact_links** — including one closing the `EVT_0009` (NFIS-II) gap identified above.

Every added record is backed by a real, dated, cited source. Full reasoning and source list:
see `../data_enrichment_log.md`.


In [12]:
import sys
sys.path.insert(0, '../src')
from enrich_data import build_new_records

new_records = build_new_records()
print(f"New records to add: {len(new_records)}")
new_records['record_type'].value_counts()


New records to add: 20


record_type
observation    12
impact_link     5
event           3
Name: count, dtype: int64

In [13]:
enriched = pd.concat([df, new_records], ignore_index=True)

print(f"Base dataset:     {len(df)} records")
print(f"New records:      {len(new_records)} records")
print(f"Enriched dataset: {len(enriched)} records")
print()
print(enriched['record_type'].value_counts())


Base dataset:     57 records
New records:      20 records
Enriched dataset: 77 records

record_type
observation    42
impact_link    19
event          13
target          3
Name: count, dtype: int64


In [14]:
# Sanity checks: no duplicate record_ids, all impact_link parent_ids resolve to real events
assert enriched['record_id'].duplicated().sum() == 0, "Duplicate record_id found!"

event_ids = set(enriched[enriched['record_type'] == 'event']['record_id'])
imp = enriched[enriched['record_type'] == 'impact_link']
orphans = imp[~imp['parent_id'].isin(event_ids)]
assert len(orphans) == 0, "Impact link(s) with missing parent event!"

print("All integrity checks passed.")


All integrity checks passed.


### 4.1 What changed: before vs. after enrichment

In [15]:
before_after = pd.DataFrame({
    'before': df['record_type'].value_counts(),
    'after': enriched['record_type'].value_counts()
}).fillna(0).astype(int)
before_after['added'] = before_after['after'] - before_after['before']
before_after


,before,after,added
record_type,,,
observation,30,42,12
impact_link,14,19,5
event,10,13,3
target,3,3,0


In [16]:
# NFIS-II now has a modeled impact_link
imp_after = enriched[enriched['record_type'] == 'impact_link']
imp_after[imp_after['parent_id'] == 'EVT_0009']


,record_id,parent_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
76,IMP_0019,EVT_0009,impact_link,NaN,ACCESS,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,ACC_OWNERSHIP,enabling,increase,medium,<NA>,36,expert,NaN,Keneni,2026-07-17,NaN,NFIS-II (EVT_0009) sets the 70% account ownership target...


## 5. Save the enriched dataset

In [17]:
enriched.to_csv('../data/processed/ethiopia_fi_unified_data_enriched.csv', index=False)
print("Saved to data/processed/ethiopia_fi_unified_data_enriched.csv")


Saved to data/processed/ethiopia_fi_unified_data_enriched.csv


## Summary

- Loaded and validated the unified-schema starter dataset (57 records: 30 observations, 10 events,
  3 targets, 14 impact_links).
- Confirmed the "no pre-assigned pillar for events" design and traced how `impact_link` connects
  events to indicators via `parent_id`.
- Found a coverage gap: NFIS-II (the policy behind the headline 70% target) had zero modeled
  impact_links.
- Enriched the dataset with 20 new, sourced records (12 observations, 3 events, 5 impact_links),
  documented in full in `data_enrichment_log.md`.
- Saved `ethiopia_fi_unified_data_enriched.csv` (77 records) as the analysis-ready dataset for Task 2.

**Next:** Task 2 — Exploratory Data Analysis on the enriched dataset.
